# Attention Value Routing Analysis

Where do values come from and go? Source decomposition,
value diversity, routing entropy, and output alignment.

## Why This Matters

Attention value routing reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_value_routing_analysis import (
    value_source_decomposition, value_diversity_per_head,
    attention_routing_entropy, value_output_alignment,
    value_routing_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Value Source Decomposition

How much does each source position contribute to the head's output?

In [ ]:
result = value_source_decomposition(model, tokens, layer=0, head=0, position=-1)
print(f"Dominant source: {result['dominant_source']}")
for s in result['per_source']:
    bar = '█' * int(s['attention_weight'] * 40)
    print(f"  Src {s['source']}: attn={s['attention_weight']:.4f}, "
          f"contrib={s['contribution_norm']:.4f} {bar}")

## Value Diversity

How diverse are value vectors across positions?

In [ ]:
for layer in range(4):
    result = value_diversity_per_head(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_diversity={result['mean_diversity']:.4f}")
    for h in result['per_head']:
        print(f"    H{h['head']}: diversity={h['diversity']:.4f}")

## Routing Entropy

How spread out is the attention routing?

In [ ]:
for layer in range(4):
    result = attention_routing_entropy(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_focused']} focused heads")
    for h in result['per_head']:
        flag = ' [FOCUSED]' if h['is_focused'] else ''
        print(f"    H{h['head']}: H={h['mean_entropy']:.4f}, "
              f"norm={h['normalized_entropy']:.4f}{flag}")

## Value Routing Summary

In [ ]:
result = value_routing_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: diversity={p['mean_diversity']:.4f}, "
          f"focused={p['n_focused']}")